# Calcul des embeddings (pas-à-pas)

Ce notebook reprend le script `src/compute_embeddings.py` pour exécuter le calcul des embeddings étape par étape.

## 1) Imports et accès au projet

In [ ]:
# Imports standard + TensorFlow/NumPy.
from pathlib import Path
import sys
import numpy as np
import tensorflow as tf

# Détecte la racine du projet (dossier qui contient src).
project_root = Path.cwd()
if (project_root / 'src').exists():
    pass
elif (project_root.parent / 'src').exists():
    project_root = project_root.parent
else:
    raise RuntimeError('Impossible de trouver le dossier src depuis le notebook.')

# Ajoute la racine projet au PYTHONPATH pour importer src/.
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print('Project root:', project_root)


## 2) Charger config et utilitaires

In [ ]:
# Utilitaires projet + fonctions du script compute_embeddings.
from src.utils import load_config, ensure_dir
from src.compute_embeddings import collect_records, build_dataset

# Charge la config YAML.
cfg = load_config(project_root / 'config.yaml')
keras_root = project_root / cfg['paths']['keras_root']
embeddings_root = project_root / cfg['paths']['embeddings_root']
models_root = project_root / cfg['paths']['models_root']

print('keras_root:', keras_root)
print('embeddings_root:', embeddings_root)
print('models_root:', models_root)


## 3) Collecter les chemins d'images et labels

In [ ]:
# Collecte des chemins d'images + labels depuis keras_root.
file_paths, labels, class_names = collect_records(keras_root)

print('Total images:', len(file_paths))
print('Nb classes:', len(class_names))
print('Exemple classe:', class_names[0])
file_paths[:3]


## 4) Construire le dataset TensorFlow

In [ ]:
# Paramètres image/batch pour le dataset tf.data.
img_size = int(cfg['model']['img_size'])
batch_size = int(cfg.get('train', {}).get('batch_size', 32))

# Construit un dataset tf.data (images + labels).
ds = build_dataset(file_paths, labels, img_size=img_size, batch_size=batch_size)
ds


## 5) Charger l'encodeur

In [ ]:
# Charge le modèle entraîné (safe_mode=False pour UnitNormalization/Lambda).
encoder = tf.keras.models.load_model(models_root / 'encoder.keras', compile=False, safe_mode=False)
encoder.summary()


## 6) Calculer les embeddings

In [ ]:
# Boucle sur les batches pour générer les embeddings.
all_embs = []
all_labels = []

for batch_imgs, batch_labels in ds:
    embs = encoder(batch_imgs, training=False).numpy()
    all_embs.append(embs)
    all_labels.append(batch_labels.numpy())

# Assemble tous les batches en un seul tableau.
embeddings = np.vstack(all_embs)
labels_out = np.concatenate(all_labels)

print('embeddings shape:', embeddings.shape)
print('labels shape:', labels_out.shape)


## 7) Sauvegarder les fichiers numpy

In [ ]:
# Crée le dossier embeddings et sauvegarde les fichiers .npy.
ensure_dir(embeddings_root)
np.save(embeddings_root / 'vectors.npy', embeddings)
np.save(embeddings_root / 'labels.npy', labels_out)
np.save(embeddings_root / 'filenames.npy', file_paths, allow_pickle=True)
np.save(embeddings_root / 'classnames.npy', class_names, allow_pickle=True)

print('Saved embeddings to:', embeddings_root.resolve())
